In [1]:
# Cell 1: Install & Import
%pip install pandas numpy requests folium global-land-mask
import os

# Create project folders
data_directories = ['data/raw', 'data/processed']
for directory in data_directories:
    if not os.path.exists(directory):
        os.makedirs(directory)
        print(f"Directory created: {directory}")
    else:
        print(f"Directory verified: {directory}")

print("✅ Environment Ready.")

Note: you may need to restart the kernel to use updated packages.
Directory verified: data/raw
Directory verified: data/processed
✅ Environment Ready.


In [2]:
# Cell 2: Data Acquisition (Session-Based & Rate Limit Aware)
import pandas as pd
import numpy as np
import requests
import time
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from io import StringIO
from datetime import datetime, timedelta, timezone

# --- CONFIGURATION ---
NASA_URL = "https://firms.modaps.eosdis.nasa.gov/data/active_fire/suomi-npp-viirs-c2/csv/SUOMI_VIIRS_C2_South_Asia_24h.csv"
WEATHER_URL = "https://api.open-meteo.com/v1/forecast"
BATCH_SIZE = 100       # Safe batch size
NORMAL_DELAY = 2    # Seconds to wait between batches
MAX_RETRIES = 3       # How many times to retry a failed batch

# Setup Session (High Speed Connection)
session = requests.Session()
retries = Retry(total=3, backoff_factor=1, status_forcelist=[500, 502, 503, 504])
session.mount('https://', HTTPAdapter(max_retries=retries))

def fetch_fire_data():
    print("Contacting NASA servers...")
    try:
        response = session.get(NASA_URL, timeout=30)
        response.raise_for_status()
        
        df = pd.read_csv(StringIO(response.text))
        
        # Spatial Filter (India Region)
        india_filter = (df['latitude'] >= 8) & (df['latitude'] <= 37) & \
                       (df['longitude'] >= 68) & (df['longitude'] <= 97)
        df_india = df[india_filter].copy()
        
        # Temporal Filter (Last 24 Hours to ensure data availability)
        df_india['acq_time'] = df_india['acq_time'].astype(str).str.zfill(4)
        df_india['dt_utc'] = pd.to_datetime(
            df_india['acq_date'] + ' ' + df_india['acq_time'], 
            format='%Y-%m-%d %H%M'
        )
        threshold = datetime.now(timezone.utc).replace(tzinfo=None) - timedelta(hours=24)
        return df_india[df_india['dt_utc'] >= threshold].copy()
    except Exception as e:
        print(f"❌ Error fetching NASA data: {e}")
        return None

# EXECUTION
fires_df = fetch_fire_data()

if fires_df is not None:
    # Prepare Data
    fires_df = fires_df[['latitude', 'longitude', 'acq_date', 'frp']].copy()
    fires_df['fire_detected'] = 1
    
    # Generate Safe Zones (Control Group)
    sample_size = max(len(fires_df), 50) # Ensure at least 50 points
    safe_df = pd.DataFrame({
        'latitude': np.random.uniform(8.0, 37.0, sample_size),
        'longitude': np.random.uniform(68.0, 97.0, sample_size),
        'acq_date': [datetime.now(timezone.utc).strftime('%Y-%m-%d')] * sample_size,
        'frp': 0.0,
        'fire_detected': 0
    })
    
    master_df = pd.concat([fires_df, safe_df], ignore_index=True)
    print(f"Total Coordinates to Process: {len(master_df)}")

    # WEATHER ENRICHMENT LOOP
    weather_data = []
    print(f"Starting Weather Enrichment (Batch Size: {BATCH_SIZE})...")
    
    for i in range(0, len(master_df), BATCH_SIZE):
        batch = master_df.iloc[i : i + BATCH_SIZE]
        params = {
            "latitude": ",".join(batch['latitude'].astype(str)),
            "longitude": ",".join(batch['longitude'].astype(str)),
            "current": "temperature_2m,relative_humidity_2m,wind_speed_10m,soil_moisture_0_to_7cm"
        }
        
        # Retry Logic for Rate Limits
        success = False
        for attempt in range(MAX_RETRIES):
            try:
                r = session.get(WEATHER_URL, params=params, timeout=30)
                if r.status_code == 429:
                    raise ValueError("Rate Limit Hit")
                r.raise_for_status()
                
                data = r.json()
                if isinstance(data, list):
                    weather_data.extend([x.get('current', {}) for x in data])
                else:
                    weather_data.append(data.get('current', {}))
                
                success = True
                time.sleep(NORMAL_DELAY) # Be polite
                break
            except Exception as e:
                wait = 20 if "Rate Limit" in str(e) else 2
                print(f"  > Batch {i} paused (Attempt {attempt+1}): {e}. Waiting {wait}s...")
                time.sleep(wait)
        
        if not success:
            print(f"  > ❌ Skipping batch {i} after failures.")
            weather_data.extend([{} for _ in range(len(batch))])
            
        if (i + BATCH_SIZE) % 100 == 0:
            print(f"Progress: {min(i + BATCH_SIZE, len(master_df))} / {len(master_df)}")

    # Final Save
    weather_df = pd.DataFrame(weather_data)
    final = pd.concat([master_df, weather_df], axis=1)
    
    # Drop rows where weather API completely failed
    final.dropna(subset=['temperature_2m'], inplace=True)
    
    final.to_csv('data/raw/master_dataset_real.csv', index=False)
    print(f"✅ Data Extraction Complete. Saved {len(final)} rows to 'data/raw/master_dataset_real.csv'")

Contacting NASA servers...
Total Coordinates to Process: 3156
Starting Weather Enrichment (Batch Size: 100)...
Progress: 100 / 3156
Progress: 200 / 3156
Progress: 300 / 3156
Progress: 400 / 3156
Progress: 500 / 3156
Progress: 600 / 3156
  > Batch 600 paused (Attempt 1): Rate Limit Hit. Waiting 20s...
  > Batch 600 paused (Attempt 2): Rate Limit Hit. Waiting 20s...
Progress: 700 / 3156
Progress: 800 / 3156
Progress: 900 / 3156
Progress: 1000 / 3156
Progress: 1100 / 3156
Progress: 1200 / 3156
  > Batch 1200 paused (Attempt 1): Rate Limit Hit. Waiting 20s...
  > Batch 1200 paused (Attempt 2): Rate Limit Hit. Waiting 20s...
Progress: 1300 / 3156
Progress: 1400 / 3156
Progress: 1500 / 3156
Progress: 1600 / 3156
Progress: 1700 / 3156
Progress: 1800 / 3156
  > Batch 1800 paused (Attempt 1): Rate Limit Hit. Waiting 20s...
  > Batch 1800 paused (Attempt 2): Rate Limit Hit. Waiting 20s...
  > Batch 1800 paused (Attempt 3): Rate Limit Hit. Waiting 20s...
  > ❌ Skipping batch 1800 after failures.


In [3]:
# Cell 3: Geospatial Validation
import folium
from global_land_mask import globe
import pandas as pd

try:
    df = pd.read_csv('data/raw/master_dataset_real.csv')
    
    # Filter for Land Only
    df['is_land'] = globe.is_land(df['latitude'], df['longitude'])
    df_final = df[df['is_land']].copy()
    
    print(f"Validation: {len(df_final)} points confirmed on land.")
    print(f"Fires: {len(df_final[df_final['fire_detected']==1])} | Safe: {len(df_final[df_final['fire_detected']==0])}")

    # Map
    m = folium.Map(location=[20.59, 78.96], zoom_start=5)
    
    # Plot Safe (Green)
    for _, row in df_final[df_final['fire_detected']==0].iterrows():
        folium.CircleMarker([row['latitude'], row['longitude']], radius=4, color='green', fill=True, fill_opacity=0.4).add_to(m)
        
    # Plot Fire (Red - On Top)
    for _, row in df_final[df_final['fire_detected']==1].iterrows():
        folium.CircleMarker([row['latitude'], row['longitude']], radius=6, color='red', fill=True, fill_color='red', fill_opacity=0.8, popup=f"FRP: {row.get('frp')}").add_to(m)
        
    m.save('data/raw/geospatial_validation_report.html')
    print("✅ Map saved to 'data/raw/geospatial_validation_report.html'")
    
except FileNotFoundError:
    print("❌ Run Cell 2 first!")

Validation: 2576 points confirmed on land.
Fires: 1578 | Safe: 998
✅ Map saved to 'data/raw/geospatial_validation_report.html'
